# GraphSAGE node prediction on the F1 KGs

Binary classification of `RaceEntry` nodes: did the entry score championship points
(`hasRacePoints`, 31.3% positive)? The same pipeline runs on all three KGs (`adhoc` /
`pure_sulo` / `hybrid_sulo`), with the same split, seeds, tuning budget and evaluation code,
so differences between them come from the KG construction alone.

The shared logic lives in `f1_node_data.py` (masking, split, features), `node_prediction/models.py`
and `node_prediction/gnn_tools.py` (training, tuning, evaluation). This notebook and its
R-GCN counterpart differ only in the model.

In [ ]:
# Must run before CUDA is initialised: avoids fragmentation OOMs across a long sweep.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "f1_population.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import torch

from f1_node_data import build_kg_data, load_population, race_entry_labels, stratified_split
from node_prediction.gnn_tools import (
    cleanup, evaluate_test_set, finished_tuning_results, limit_cuda_memory_to_free_vram,
    run_sequential_optimization_with_verification, runs_ended_by_epoch_cap, set_seed, train_model,
)
from node_prediction.models import GraphSAGEModel

KGS = ["adhoc", "pure_sulo", "hybrid_sulo"]
OUT_DIR = ROOT / "outputs" / "f1" / "node_prediction" / "graphsage"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(torch.__version__, "| CUDA:", torch.cuda.is_available())
# WSL2 would otherwise overcommit VRAM into RAM and fail with a driver error instead of an OOM,
# which the sweep could not skip.
if torch.cuda.is_available():
    # Close other GPU programs first: the cap is what is free now, and the largest config peaks at
    # 9.2 GiB (R-GCN; GraphSAGE 4.2 GiB), so a lower cap OOMs it, which the tuning cell then reports.
    print(f"CUDA memory cap: {limit_cuda_memory_to_free_vram() / 2**30:.2f} GiB")

## Data

`build_kg_data` renders each KG from `data/f1/assertions.csv` **without** the label and the
race outcomes (`hasFinishPosition`, `hasStatus`, `hasLapsCompleted`), for every `RaceEntry`.
Masking happens before rendering, so Pure SULO's intermediate-node paths disappear completely,
and where a start and finish position were one merged triple, the start position survives.
Labels come from the assertions, never from a graph.

The 80/10/10 split is stratified by label and computed once on IRIs, so all KGs share it.
Node features never read IRI text: an instance is described by its `rdf:type` set, a class node
by being that class, a literal by its datatype and (standardised) value or, for strings, by a
learnable embedding of the string. The degree feature is computed on the masked graph.

In [ ]:
entities, assertions = load_population(ROOT / "data" / "f1")
labels = race_entry_labels(assertions)
split = stratified_split(labels, seed=42)

kg_data = {kg: build_kg_data(kg, entities, assertions, split) for kg in KGS}

pd.DataFrame({
    kg: {
        "nodes": d.data.num_nodes,
        "edges (fwd + inv)": d.data.edge_index.size(1),
        "relation types (fwd + inv)": d.num_relations,
        "feature keys": len(d.vocabulary),
        "train / val / test": f"{len(d.train)} / {len(d.val)} / {len(d.test)}",
        "positive rate (train)": round(d.data.y[d.train].mean().item(), 3),
    }
    for kg, d in kg_data.items()
}).T

## Tuning constants

Every value below must be set, with a source, before anything is trained: the grids are
not carried over from the CAC phase. The same values are used for every KG.
GraphSAGE has no relation types, so no basis decomposition is tuned (`grid_bases=[None]`).

In [ ]:
# Hidden dims: Hamilton et al. 2017 use 256-d outputs at every depth and 512/1024 pooling dims;
# Schlichtkrull et al. 2018 use 16 hidden units for entity classification.
#   https://arxiv.org/abs/1706.02216
#   https://arxiv.org/abs/1703.06103
# Full-batch 512-d fits on the 12 GB RTX 3060 for every KG (GraphSAGE peak 4.2 GiB: Pure SULO, 3 layers).
GRID_1A_DIM = [32, 64, 128, 256, 512]

# Layers (full-batch, so no neighbour fanouts): 1 vs 2 = Hamilton et al. 2017, Section 4.3 (K=2 gave
# ~10-15% more than K=1, K>2 only 0-5% more); 2 = Schlichtkrull et al. 2018; 3 = the depth of PyG's
# GraphSAGE ogbn-products example. A RaceEntry reaches its grid rank in 2 hops in Ad-hoc and in 3 in
# Hybrid/Pure.
#   https://arxiv.org/abs/1703.06103
#   https://arxiv.org/abs/1706.02216
#   https://github.com/pyg-team/pytorch_geometric/blob/2.3.0/examples/ogbn_products_sage.py
GRID_1B_DEPTH = [1, 2, 3]

# Learning rate: Hamilton et al. 2017 searched {1e-2, 1e-3, 1e-4} (supervised); Schlichtkrull et al. 2018
# and PyG's R-GCN example use 1e-2.
#   https://arxiv.org/abs/1706.02216
#   https://arxiv.org/abs/1703.06103
#   https://github.com/pyg-team/pytorch_geometric/blob/master/examples/rgcn_link_pred.py
GRID_2B_LR = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2]

# Weight decay (Adam): Schlichtkrull et al. 2018 searched L2 in {0, 5e-4}; GraphGym (You et al. 2020)
# defaults to 5e-4.
#   https://arxiv.org/abs/1703.06103
#   https://github.com/pyg-team/pytorch_geometric/blob/master/torch_geometric/graphgym/config.py
GRID_2C_WD = [0, 1e-5, 1e-4, 5e-4, 1e-3]

# Defaults while earlier phases run (phase 1 tunes depth jointly with dim, so DEFAULT_DEPTH is only
# used by the shuffled-label check): 2 layers = Schlichtkrull et al. 2018 / Hamilton et al. 2017 (K=2);
# lr 1e-2 = Kipf & Welling 2017 and Schlichtkrull et al. 2018, both full-batch with Adam.
# wd 1e-4 has no direct literature value (between R-GCN's 0 and 5e-4).
#   https://arxiv.org/abs/1609.02907
#   https://arxiv.org/abs/1703.06103
DEFAULT_DEPTH = 2
DEFAULT_LR = 1e-2
DEFAULT_WD = 1e-4

# Full-batch, so one epoch = one gradient step. Max epochs 400 = the top of GraphGym's design space
# {100, 200, 400} (You et al. 2020, Section 4); Kipf & Welling 2017 cap at 200, but the Ad-hoc R-GCN
# (3 layers) was still improving at epoch 200. Patience 50: no literature value (user choice,
# 2026-10-21). Kipf & Welling's window of 10 (on val loss) stopped runs on val-ROC-AUC plateaus,
# e.g. Ad-hoc 3 layers at 0.755, which reaches 0.837 by epoch 200. The final retrain uses
# best_epoch + 1 epochs regardless.
#   https://arxiv.org/abs/2011.08843
#   https://arxiv.org/abs/1609.02907
EPOCHS = 400
PATIENCE = 50

# Node input embedding size: the GraphSAGE reference implementation recommends learned identity
# features of 64-256 dims (README, --identity_dim).
#   https://github.com/williamleif/GraphSAGE#readme
FEATURE_DIM = 256

# Dropout: PyG's GraphSAGE ogbn-products example uses p=0.5; Hamilton et al. 2017 do not
# report dropout.
#   https://github.com/pyg-team/pytorch_geometric/blob/2.3.0/examples/ogbn_products_sage.py
DROPOUT = 0.5

# Every tuning config is trained once per seed and scored by its mean val ROC-AUC, so a seed that
# collapses early lowers a config's score instead of deciding it; the final retrain uses the
# median per-seed best epoch. 3 seeds: user choice (2026-10-24), no literature value.
TUNING_SEEDS = [0, 1, 2]
FINAL_SEEDS = [0, 1, 2, 3, 4]   # final retrain + test, reported as mean ± std

missing = [name for name, value in dict(globals()).items() if name.isupper() and value is None]
assert not missing, f"Set these before training: {missing}"

In [ ]:
def model_factory(d):
    """(hidden_list, bases) -> fresh model for one KG; everything KG-specific is bound here."""
    def build(hidden_list, bases):
        return GraphSAGEModel(
            num_keys=len(d.vocabulary), feature_dim=FEATURE_DIM, hidden_channels_list=hidden_list,
            num_relations=d.num_relations, num_bases=bases, dropout=DROPOUT,
        )
    return build

def bases_grid(d):
    return [None]  # GraphSAGE has no relation types

def default_bases(d):
    return None

## Pipeline check: shuffled labels

Trains the default configuration (smallest grid dim) on each real graph with the labels permuted across all
labelled entries. The test ROC-AUC must stay at chance (≈ 0.5): otherwise the target vector
reaches the model's inputs somewhere in the pipeline. (Leakage of the *true* outcome through
graph facts is handled by the masking above and checked in `tests/test_f1_node_data.py`.)

In [ ]:
control = {}
for kg, d in kg_data.items():
    labelled = torch.cat([d.train, d.val, d.test])
    shuffled = d.data.clone()
    shuffled.y[labelled] = d.data.y[labelled][torch.randperm(len(labelled), generator=torch.Generator().manual_seed(0))]

    set_seed(TUNING_SEEDS[0])
    model = model_factory(d)([GRID_1A_DIM[0]] * (DEFAULT_DEPTH - 1), default_bases(d))
    model, _ = train_model(model, shuffled, d.train, d.val, epochs=EPOCHS, lr=DEFAULT_LR,
                           weight_decay=DEFAULT_WD, patience=PATIENCE)
    control[kg] = evaluate_test_set(model, shuffled, d.test)["roc_auc"]
    model.cpu()
    del model, shuffled
    cleanup()

control

## Sequential tuning and seeded final evaluation

Per KG: a joint dim × depth grid, then greedy bases → lr → wd at the winning architecture, then a
2×2 verification grid over the top-2 dims (at the winning depth) × the top-2 regularisation combos.
Depth and dim are tuned jointly because the depth a KG needs decides which dims can be compared: in
Pure/Hybrid SULO the grid rank is 3 hops from a `RaceEntry`, so dims ranked at 2 layers were ranked
without it. Every configuration is trained once per seed in `TUNING_SEEDS` and scored by its mean
val ROC-AUC (per-seed values are in the CSV). The winner is retrained on train+val for its median
best epoch + 1 epochs and tested once per seed in `FINAL_SEEDS`.
Training and evaluation are full-batch: every layer aggregates each node's complete
neighbourhood, with no neighbour sampling, so validation and test scores are deterministic.
Sampling hid the identity of shared hubs: a `Position/<n>` node has ~1,800 neighbours and a single
edge towards its rank, which a fanout of 10 almost never drew. The sampled R-GCN run
(`outputs/f1/node_prediction/rgcn_sampled/`) scored below ranking entries by grid position alone.

In [ ]:
tuning = {}
for kg, d in kg_data.items():
    csv_path = OUT_DIR / f"{kg}_sequential_tuning_results.csv"
    # Resume after a crash: a KG whose search already finished with these grids and seeds is loaded, not re-run.
    tuning[kg] = finished_tuning_results(csv_path, GRID_1A_DIM, GRID_1B_DEPTH, TUNING_SEEDS, FINAL_SEEDS)
    if tuning[kg] is not None:
        print(f"{kg}: finished results loaded from {csv_path}")
        continue
    tuning[kg] = run_sequential_optimization_with_verification(
        model_factory=model_factory(d), graph_data=d.data,
        train_indices=d.train, val_indices=d.val, test_indices=d.test,
        schema_label=kg, save_dir=f"{OUT_DIR / kg}/", csv_path=csv_path,
        grid_dim=GRID_1A_DIM, grid_depth=GRID_1B_DEPTH, grid_bases=bases_grid(d), default_bases=default_bases(d),
        grid_lr=GRID_2B_LR, grid_wd=GRID_2C_WD,
        default_lr=DEFAULT_LR, default_wd=DEFAULT_WD,
        epochs=EPOCHS, patience=PATIENCE, tuning_seeds=TUNING_SEEDS, final_seeds=FINAL_SEEDS,
    )
    cleanup()

# OOM-ed configs are skipped, not fatal: if any KG lost one, the KGs were not tuned over the same grid.
failed = pd.concat(df[df["status"] != "ok"].assign(kg=kg) for kg, df in tuning.items())
assert failed.empty, f"Tuning runs that did not finish:\n{failed[['kg', 'phase', 'parameter_tested', 'status']]}"

## Results

Test metrics of the final model per KG, mean ± std over `FINAL_SEEDS`.

In [ ]:
final = pd.concat(
    df[df["phase"] == "4_Final_Test_Evaluation"].assign(kg=kg) for kg, df in tuning.items()
)
metrics = ["test_roc_auc", "test_pr_auc", "test_f1", "test_accuracy", "test_class0_f1", "test_class1_f1"]
summary = final.groupby("kg")[metrics].agg(["mean", "std"]).round(4)
summary.to_csv(OUT_DIR / "final_test_summary.csv")
summary

## Limitation: runs ended by the epoch cap

A run stops early at `best_epoch + PATIENCE`, so a seed whose best epoch is ≥ `EPOCHS − PATIENCE`
was cut off by the `EPOCHS` cap while possibly still improving. Such configurations may be
under-trained and their val ROC-AUC understated, which matters most when it is a KG's winner.
`EPOCHS = 400` is kept at the top of GraphGym's design
space rather than raised without a source, so this is reported as a limitation. The table lists
every trained tuning configuration with at least one capped seed; `best_epoch` of a phase-4 row is
the median of its winner's seeds.

In [ ]:
capped = pd.concat(
    runs_ended_by_epoch_cap(df, EPOCHS, PATIENCE).assign(kg=kg) for kg, df in tuning.items()
)
winners = final.drop_duplicates("kg")[["kg", "hidden_dim", "layers", "num_bases", "lr", "weight_decay", "best_epoch"]]
display(winners.assign(winner_capped=winners["best_epoch"] >= EPOCHS - PATIENCE))
capped[["kg", "phase", "parameter_tested", "val_roc_auc", "val_roc_auc_seeds", "best_epoch_seeds"]]